In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
import random

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

BASE_URL = "https://raw.githubusercontent.com/carterprince/spotify-classification/master/data/"

def load_data():
    print("Loading data from GitHub...")

    X_train = pd.read_csv(BASE_URL + 'X_train.csv').values
    y_train = pd.read_csv(BASE_URL + 'y_train.csv').values.flatten()
    X_test  = pd.read_csv(BASE_URL + 'X_test.csv').values
    y_test  = pd.read_csv(BASE_URL + 'y_test.csv').values.flatten()

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data()

hyperparameter_grid = [
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 20, "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 500, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
]

trials = []
best_accuracy = -1
best_hyperparameters = None
best_confusion_matrix = None

print(f"Running {len(hyperparameter_grid)} trials...\n")

for params in hyperparameter_grid:
    print(f"Testing parameters: {params}")

    model = RandomForestClassifier(**params, random_state=42)

    start_train = time.time()
    model.fit(X_train, y_train)
    end_train = time.time()

    start_test = time.time()
    y_pred = model.predict(X_test)
    end_test = time.time()

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print(f"   --> Accuracy: {acc:.4f}")
    print(f"   --> Train time: {end_train - start_train:.2f}s, Test time: {end_test - start_test:.2f}s\n")

    trials.append({
        "hyperparameters": params,
        "confusion_matrix": cm.tolist(),
        "accuracy": acc
    })

    if acc > best_accuracy:
        best_accuracy = acc
        best_hyperparameters = params
        best_confusion_matrix = cm



Loading data from GitHub...
Running 4 trials...

Testing parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
   --> Accuracy: 0.5003
   --> Train time: 13.71s, Test time: 0.14s

Testing parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
   --> Accuracy: 0.5277
   --> Train time: 26.39s, Test time: 0.37s

Testing parameters: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2}
   --> Accuracy: 0.5307
   --> Train time: 42.59s, Test time: 0.69s

Testing parameters: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
   --> Accuracy: 0.5386
   --> Train time: 97.03s, Test time: 1.49s



In [3]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_data = {
    "model_name": "Random Forest Classifier",
    "person_name": "Anvita Yerramsetty",
    "best_hyperparameters": best_hyperparameters,
    "best_confusion_matrix": best_confusion_matrix.tolist(),
    "trials": trials,
    "total_train_time": round(end_train - start_train, 4),
    "total_test_time": round(end_test - start_test, 4)
}

output_path = os.path.join(OUTPUT_DIR, "random_forest.json")
with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"\n SAVED JSON TO: {output_path}")


 SAVED JSON TO: output/random_forest.json
